In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !tar -xvf './cifar100.tar'

#Step 1: Download a dataset and preview images


# Step 2: Custom Data Loading

In [10]:
import os
import time
import glob
import torch
import shutil
import torchvision
import torch.nn as nn
from torch.nn import functional as F
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

config = {
    "data_path":"./cifar100",
    "batch_size":128,
}

class mydataset(Dataset):
    def __init__(self, data_dir, flag, transform):
        super(mydataset,self).__init__()
        self.root       = data_dir
        self.label      = flag
        self.transform  = transform

        self.img_dir = os.path.join(self.root, self.label)
        self.img_names  = glob.glob(os.path.join(self.img_dir, '*.jpg'))

        self.tags = ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm']

    def RGB2Gradient(self, img:torch.tensor):
        """
        Converts an RGB image tensor to its gradient magnitude using Sobel operator.
        The output is replicated to 3 channels to match the input dimensions of the other branch.
        """
        
        # Define Sobel kernels
        sobel_kernel_x = torch.tensor([[-1., 0., 1.], 
                                       [-2., 0., 2.], 
                                       [-1., 0., 1.]], dtype=torch.float32).reshape((1, 1, 3, 3))
        sobel_kernel_y = torch.tensor([[-1., -2., -1.], 
                                       [ 0.,  0.,  0.], 
                                       [ 1.,  2.,  1.]], dtype=torch.float32).reshape((1, 1, 3, 3))

        sobel_kernel_x = sobel_kernel_x.to(img.device)
        sobel_kernel_y = sobel_kernel_y.to(img.device)

        # Convert to grayscale: [3, H, W] -> Output [1, H, W]
        # gray_img = img[0, :, :] * 0.2989 + img[1, :, :] * 0.5870 + img[2, :, :] * 0.1140
        # gray_img = gray_img.unsqueeze(0)
        gray_img = transforms.Grayscale(num_output_channels=1)(img)

        # Add batch dimension [1, H, W] -> [1, 1, H, W]
        gray_img_batch = gray_img.unsqueeze(0) 

        # [YOU NEED TO FILL] Apply Sobel filters
        # Use F.conv2d, gray_img_batch, and the Sobel kernels (sobel_kernel_x, sobel_kernel_y) 
        # to calculate the gradients in the x and y directions.
        grad_x = F.conv2d(gray_img_batch,sobel_kernel_x, bias=None, stride = 1, padding = 1)
        grad_y = F.conv2d(gray_img_batch,sobel_kernel_y, bias=None, stride = 1, padding = 1)
        
        # [YOU NEED TO FILL] Calculate gradient magnitude ---
        # Calculate the magnitude (G = sqrt(Gx^2 + Gy^2)) from grad_x and grad_y.
        # The result should be stored in a variable named 'magnitude'.
        magnitude = (grad_x*grad_x+grad_y*grad_y)**0.5

        # Normalize magnitude
        mag_min = magnitude.min()
        mag_max = magnitude.max()
        epsilon = 1e-6 # Avoid division by zero
        normalized_magnitude = (magnitude - mag_min) / (mag_max - mag_min + epsilon)
        
        # Replicate to 3 channels [1, 1, H, W] -> [1, 3, H, W]
        normalized_magnitude_3channel = normalized_magnitude.repeat(1, 3, 1, 1)

        # Remove batch dimension [1, 3, H, W] -> [3, H, W]
        return normalized_magnitude_3channel.squeeze(0)

    def __getitem__(self,idx):
        img_name = self.img_names[idx]
        with Image.open(img_name) as image:
            img = self.transform(image.convert("RGB"))

        # Match the complete class name: beetle != bee, woman != man.
        class_name = os.path.splitext(os.path.basename(img_name))[0].rsplit("_", 1)[0]
        tag = self.tags.index(class_name)
        imgGradient=self.RGB2Gradient(img)
        return img, imgGradient, tag
    def __len__(self):
        return len(self.img_names)

transform_train = transforms.Compose(
        [transforms.Resize([32, 32]),
         transforms.ToTensor()])
transform_test = transforms.Compose(
    [transforms.Resize([32, 32]),
      transforms.ToTensor()])

import random
from torch.utils.data import Subset

split_seed = 42
val_ratio = 0.1

# Separate transforms for training and validation, using the same files on disk.
train_source = mydataset(config["data_path"], "train", transform_train)
val_source = mydataset(config["data_path"], "train", transform_test)
train_source.img_names.sort()
val_source.img_names = train_source.img_names

# Stratify by complete class name, then keep this split fixed across epochs.
class_indices = {}
for index, filename in enumerate(train_source.img_names):
    class_name = os.path.splitext(os.path.basename(filename))[0].rsplit("_", 1)[0]
    class_indices.setdefault(class_name, []).append(index)

rng = random.Random(split_seed)
train_indices, val_indices = [], []
for indices in class_indices.values():
    assert len(indices) >= 2, "Each class needs training and validation samples"
    rng.shuffle(indices)
    val_count = max(1, min(len(indices) - 1, int(len(indices) * val_ratio)))
    val_indices.extend(indices[:val_count])
    train_indices.extend(indices[val_count:])

assert set(train_indices).isdisjoint(val_indices)
assert len(train_indices) + len(val_indices) == len(train_source)
train_dataset = Subset(train_source, train_indices)
val_dataset = Subset(val_source, val_indices)
test_dataset = mydataset(config["data_path"], "test", transform_test)
split_info = {
    "seed": split_seed,
    "val_ratio": val_ratio,
    "train_indices": train_indices,
    "val_indices": val_indices,
}

train_loader = DataLoader(
    train_dataset, batch_size=config["batch_size"], shuffle=True,
    num_workers=4, pin_memory=True, drop_last=False)
val_loader = DataLoader(
    val_dataset, batch_size=config["batch_size"], shuffle=False,
    num_workers=4, pin_memory=True, drop_last=False)
test_loader = DataLoader(
    test_dataset, batch_size=config["batch_size"], shuffle=False,
    num_workers=4, pin_memory=True, drop_last=False)

print(f"train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")


train=45000, val=5000, test=10000


# Step 3: Configure the Neural Network

In [11]:
import torch.nn as nn
from torch.nn import functional as F
from torch.nn import Sequential as Seq

class CommonBlock(nn.Module):
    def __init__(self, in_channel, out_channel, stride):        # 普通Block简单完成两次卷积操作
        super(CommonBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(out_channel, out_channel, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)

    def forward(self, x):
        identity = x                                            # 普通Block的shortcut为直连，不需要升维下采样

        x = F.relu(self.bn1(self.conv1(x)), inplace=True)       # 完成一次卷积
        x = self.bn2(self.conv2(x))                             # 第二次卷积不加relu激活函数

        x += identity                                           # 两路相加
        return F.relu(x, inplace=True)                          # 添加激活函数输出

class SpecialBlock(nn.Module):                                  # 特殊Block完成两次卷积操作，以及一次升维下采样
    def __init__(self, in_channel, out_channel, stride):        # 注意这里的stride传入一个数组，shortcut和残差部分stride不同
        super(SpecialBlock, self).__init__()
        self.change_channel = nn.Sequential(                    # 负责升维下采样的卷积网络change_channel
            nn.Conv2d(in_channel, out_channel, kernel_size=1, stride=stride[0], padding=0, bias=False),
            nn.BatchNorm2d(out_channel)
        )
        self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=3, stride=stride[0], padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(out_channel, out_channel, kernel_size=3, stride=stride[1], padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)

    def forward(self, x):
        identity = self.change_channel(x)                       # 调用change_channel对输入修改，为后面相加做变换准备

        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.bn2(self.conv2(x))                             # 完成残差部分的卷积

        x += identity
        return F.relu(x, inplace=True)                          # 输出卷积单元

class ConvNet(nn.Module):
    def __init__(self, classes_num):
        super(ConvNet, self).__init__()
        self.prepare1 = nn.Sequential(           # 预处理==》[batch, 64, 56, 56]
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1)
        )
        self.prepare2 = nn.Sequential(           # 预处理==》[batch, 64, 56, 56]
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1)
        )
        self.layer1 = nn.Sequential(            # layer1有点特别，由于输入输出的channel均是64，故两个CommonBlock
            CommonBlock(64, 64, 1),
            CommonBlock(64, 64, 1)
        )
        self.layer2 = nn.Sequential(            # layer234类似，由于输入输出的channel不同，故一个SpecialBlock，一个CommonBlock
            SpecialBlock(64, 128, [2, 1]),
            CommonBlock(128, 128, 1)
        )
        self.layer3 = nn.Sequential(
            SpecialBlock(128, 256, [2, 1]),
            CommonBlock(256, 256, 1)
        )
        self.layer4 = nn.Sequential(
            SpecialBlock(256, 512, [2, 1]),
            CommonBlock(512, 512, 1)
        )
        self.head = Seq(nn.AdaptiveAvgPool2d(1),
                  nn.Flatten(start_dim=1),
                  nn.Linear(512, classes_num))
    def forward(self, x, y):
        x = self.prepare1(x) + self.prepare2(y)         # 预处理
        
        x = self.layer1(x)          # 四个卷积单元
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.head(x)

        return x

class CrossAttention(nn.Module):
    """Bidirectional RGB/gradient attention followed by residual feature fusion."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.rgb_norm = nn.LayerNorm(channels)
        self.gradient_norm = nn.LayerNorm(channels)
        self.rgb_attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.gradient_attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, rgb, gradient):
        if rgb.ndim != 4 or rgb.shape != gradient.shape:
            raise ValueError("CrossAttention expects equally shaped [B, C, H, W] features")
        batch, channels, height, width = rgb.shape
        # Each spatial position is a token: [B, C, H, W] -> [B, H*W, C].
        rgb_tokens = rgb.flatten(2).transpose(1, 2)
        gradient_tokens = gradient.flatten(2).transpose(1, 2)
        rgb_norm = self.rgb_norm(rgb_tokens)
        gradient_norm = self.gradient_norm(gradient_tokens)

        # RGB queries gradient keys/values; gradient queries RGB keys/values.
        rgb_update, _ = self.rgb_attention(
            rgb_norm, gradient_norm, gradient_norm, need_weights=False)
        gradient_update, _ = self.gradient_attention(
            gradient_norm, rgb_norm, rgb_norm, need_weights=False)
        fused = (rgb_tokens + rgb_update + gradient_tokens + gradient_update) / 2
        return fused.transpose(1, 2).reshape(batch, channels, height, width)


class DualConvNet(nn.Module):
    def __init__(self, classes_num):
        super(DualConvNet, self).__init__()
        self.prepare = nn.Sequential(           # [B, 3, 64, 64] -> [B, 64, 16, 16]
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1)
        )
        self.layer1 = nn.Sequential(            # layer1有点特别，由于输入输出的channel均是64，故两个CommonBlock
            CommonBlock(64, 64, 1),
            CommonBlock(64, 64, 1)
        )
        self.layer2 = nn.Sequential(            # layer234类似，由于输入输出的channel不同，故一个SpecialBlock，一个CommonBlock
            SpecialBlock(64, 128, [2, 1]),
            CommonBlock(128, 128, 1)
        )
        self.layer3 = nn.Sequential(
            SpecialBlock(128, 256, [2, 1]),
            CommonBlock(256, 256, 1)
        )

        self.prepareRGB = nn.Sequential(           # [B, 3, 64, 64] -> [B, 64, 16, 16]
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1)
        )
        self.layerRGB1 = nn.Sequential(            # layer1有点特别，由于输入输出的channel均是64，故两个CommonBlock
            CommonBlock(64, 64, 1),
            CommonBlock(64, 64, 1)
        )
        self.layerRGB2 = nn.Sequential(            # layer234类似，由于输入输出的channel不同，故一个SpecialBlock，一个CommonBlock
            SpecialBlock(64, 128, [2, 1]),
            CommonBlock(128, 128, 1)
        )
        self.layerRGB3 = nn.Sequential(
            SpecialBlock(128, 256, [2, 1]),
            CommonBlock(256, 256, 1)
        )

        # add cross_attention: layer3 outputs [B, 256, 4, 4] for 64x64 inputs.
        self.cross_attention = CrossAttention(channels=256, num_heads=4)
        
        self.layer4 = nn.Sequential(
            SpecialBlock(256, 512, [2, 1]),
            CommonBlock(512, 512, 1)
        )

        
        self.head = Seq(nn.AdaptiveAvgPool2d(1),
                  nn.Flatten(start_dim=1),
                  nn.Linear(512, classes_num))
    def forward(self, x, y):
        # x: original RGB image; y: its gradient magnitude image.
        x = self.prepareRGB(x)
        x = self.layerRGB1(x)
        x = self.layerRGB2(x)
        x = self.layerRGB3(x)

        y = self.prepare(y)
        y = self.layer1(y)
        y = self.layer2(y)
        y = self.layer3(y)

        # add cross_attention; the fused map keeps its 256 channels.
        x = self.cross_attention(x, y)
        x = self.layer4(x)
        return self.head(x)

class WRNDualConvNet(nn.Module):
    """WRN-style stage widths/depth; reuse the existing post-activation blocks."""
    def __init__(self, classes_num, blocks_per_stage=4):
        super(WRNDualConvNet, self).__init__()
        # Channel constants: change widths here; every dependent layer uses them.
        channel = 16
        channel1 = 32
        channel2 = 64
        channel3 = 128
        if blocks_per_stage < 1:
            raise ValueError("blocks_per_stage must be at least 1")

        # Gradient branch: 32x32 -> 32x32 -> 16x16 -> 8x8.
        self.prepare = nn.Sequential(
            nn.Conv2d(3, channel, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channel),
            nn.ReLU(inplace=True)
        )
        self.layer1 = nn.Sequential(
            SpecialBlock(channel, channel1, [1, 1]),
            *[CommonBlock(channel1, channel1, 1) for _ in range(blocks_per_stage - 1)]
        )
        self.layer2 = nn.Sequential(
            SpecialBlock(channel1, channel2, [2, 1]),
            *[CommonBlock(channel2, channel2, 1) for _ in range(blocks_per_stage - 1)]
        )
        self.layer3 = nn.Sequential(
            SpecialBlock(channel2, channel3, [2, 1]),
            *[CommonBlock(channel3, channel3, 1) for _ in range(blocks_per_stage - 1)]
        )

        # RGB branch: same layout, separately constructed parameters.
        self.prepareRGB = nn.Sequential(
            nn.Conv2d(3, channel, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channel),
            nn.ReLU(inplace=True)
        )
        self.layerRGB1 = nn.Sequential(
            SpecialBlock(channel, channel1, [1, 1]),
            *[CommonBlock(channel1, channel1, 1) for _ in range(blocks_per_stage - 1)]
        )
        self.layerRGB2 = nn.Sequential(
            SpecialBlock(channel1, channel2, [2, 1]),
            *[CommonBlock(channel2, channel2, 1) for _ in range(blocks_per_stage - 1)]
        )
        self.layerRGB3 = nn.Sequential(
            SpecialBlock(channel2, channel3, [2, 1]),
            *[CommonBlock(channel3, channel3, 1) for _ in range(blocks_per_stage - 1)]
        )

        # Fuse at [B, channel3, 8, 8], then pool and classify directly.
        self.cross_attention = CrossAttention(channels=channel3, num_heads=4)
        self.head = Seq(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(start_dim=1),
            nn.Linear(channel3, classes_num)
        )

    def forward(self, x, y):
        # x: original RGB image; y: its gradient magnitude image.
        x = self.prepareRGB(x)
        x = self.layerRGB1(x)
        x = self.layerRGB2(x)
        x = self.layerRGB3(x)

        y = self.prepare(y)
        y = self.layer1(y)
        y = self.layer2(y)
        y = self.layer3(y)

        x = self.cross_attention(x, y)
        return self.head(x)


config = {
    "lr":1e-2,
    "momentum":0.9,
    "weight_decay":1e-4,
}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
net = WRNDualConvNet(classes_num=100, blocks_per_stage=4).to(device)
print(f"{type(net).__name__}: {sum(p.numel() for p in net.parameters()):,} parameters")
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(net.parameters(), lr=config["lr"], momentum=config["momentum"], weight_decay=config["weight_decay"])

cuda:0
WRNDualConvNet: 3,079,044 parameters


# Step 4: Train the network and save model

In [12]:
import time

class AverageMeter(object):
    def __init__(self):
      self.reset()

    def reset(self):
      self.val   = 0
      self.avg   = 0
      self.sum   = 0
      self.count = 0

    def update(self, val, n=1):
      self.val   = val
      self.sum   += val * n
      self.count += n
      self.avg   = self.sum / self.count

def accuracy(output, target, topk=(1,1)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred    = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
      correct_k = correct[:k].view(-1).float().sum(0)
      res.append(correct_k.mul_(100.0 / batch_size))
    return res[0]

def train(train_loader, net, optimizer, criterion, epoch):
    batch_time = AverageMeter()
    data_time  = AverageMeter()
    top1       = AverageMeter()

    LOSS = AverageMeter()

    net.train()

    end = time.time()
    for i, (img, imgGradient, target) in enumerate(train_loader, start=1):
        data_time.update(time.time() - end)

        img = img.to(device)
        imgGradient = imgGradient.to(device)
        target = target.to(device)

        out = net(img, imgGradient)

        loss = criterion(out, target)

        prec1 = accuracy(out, target) # prec1:list
        top1.update(prec1.item(), img.size(0))

        LOSS.update(loss.item(), img.size(0))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_time.update(time.time() - end)
        end = time.time()

        if i % 50 == 1:
            log_str = ('Epoch[{0}]:[{1:03}/{2:03}] '
                       'Time:{batch_time.val:.4f} '
                       'Data:{data_time.val:.4f}  '
                       'loss:{loss.val:.4f}({loss.avg:.4f})  '
                       'prec@1:{top1.val:.2f}({top1.avg:.2f})  '.format(
                       epoch, i, len(train_loader), batch_time=batch_time, data_time=data_time,
                       loss=LOSS,
                       top1=top1))
            print(log_str)

    return LOSS.avg

@torch.no_grad()
def validate(val_loader, net, criterion):
    net.eval()
    losses = AverageMeter()
    top1 = AverageMeter()

    for img, imgGradient, target in val_loader:
        img = img.to(device)
        imgGradient = imgGradient.to(device)
        target = target.to(device)
        out = net(img, imgGradient)
        loss = criterion(out, target)
        prec1 = accuracy(out, target)
        losses.update(loss.item(), img.size(0))
        top1.update(prec1.item(), img.size(0))

    return losses.avg, top1.avg


def save_checkpoint(state, is_best, save_root):
    os.makedirs(save_root, exist_ok=True)
    last_path = os.path.join(save_root, "last.pt")
    # Replace only after writing succeeds, preserving the previous checkpoint on failure.
    torch.save(state, last_path + ".tmp")
    os.replace(last_path + ".tmp", last_path)
    if is_best:
        best_path = os.path.join(save_root, "best.pt")
        shutil.copyfile(last_path, best_path + ".tmp")
        os.replace(best_path + ".tmp", best_path)


config = {
    "save_root": "./result/wrn_dual_cross_attention",
    "epochs": 30,
}

best_val_acc = -1.0
best_epoch = 0
for epoch in range(1, config["epochs"] + 1):
    epoch_start_time = time.time()
    train_loss = train(train_loader, net, optimizer, criterion, epoch)
    val_loss, val_acc = validate(val_loader, net, criterion)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_epoch = epoch

    save_checkpoint({
        "epoch": epoch,
        "architecture": type(net).__name__,
        "net": net.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "split": split_info,
    }, is_best, config["save_root"])

    print(
        f"Epoch {epoch}/{config['epochs']} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.2f}% | "
        f"best={best_val_acc:.2f}% at epoch {best_epoch} | "
        f"time={time.time() - epoch_start_time:.1f}s"
    )

# Single-image prediction and Step 6 now use the best validation checkpoint.
best_path = os.path.join(config["save_root"], "best.pt")
checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
assert checkpoint["architecture"] == type(net).__name__
net.load_state_dict(checkpoint["net"])
net.eval()
print(
    f"Loaded best.pt: epoch={checkpoint['epoch']}, "
    f"val_acc={checkpoint['val_acc']:.2f}%"
)


Epoch[1]:[001/352] Time:0.7965 Data:0.5323  loss:4.8130(4.8130)  prec@1:2.34(2.34)  
Epoch[1]:[051/352] Time:0.0334 Data:0.0001  loss:4.2729(4.4604)  prec@1:2.34(3.57)  
Epoch[1]:[101/352] Time:0.0309 Data:0.0001  loss:3.7058(4.2578)  prec@1:12.50(5.66)  
Epoch[1]:[151/352] Time:0.0301 Data:0.0001  loss:3.6717(4.1063)  prec@1:10.16(7.27)  
Epoch[1]:[201/352] Time:0.0321 Data:0.0001  loss:3.5104(3.9820)  prec@1:15.62(8.78)  
Epoch[1]:[251/352] Time:0.0262 Data:0.0001  loss:3.3868(3.8705)  prec@1:14.84(10.30)  
Epoch[1]:[301/352] Time:0.0363 Data:0.0001  loss:3.2158(3.7798)  prec@1:20.31(11.65)  
Epoch[1]:[351/352] Time:0.0430 Data:0.0000  loss:3.0455(3.6998)  prec@1:28.12(12.90)  
Epoch 1/30 | train_loss=3.6996 | val_loss=3.6556 | val_acc=14.74% | best=14.74% at epoch 1 | time=28.6s
Epoch[2]:[001/352] Time:0.4062 Data:0.3852  loss:3.0266(3.0266)  prec@1:25.00(25.00)  
Epoch[2]:[051/352] Time:0.0401 Data:0.0001  loss:3.3314(3.0552)  prec@1:22.66(23.47)  
Epoch[2]:[101/352] Time:0.0906 Da

KeyboardInterrupt: 

# Step 5: Test on single image

In [ ]:
with Image.open("./cifar100/test/apple_9904.jpg") as image:
    img = transform_test(image.convert("RGB"))
# RGB2Gradient takes one [3, H, W] image, before adding the batch dimension.
imgGradient = test_dataset.RGB2Gradient(img)
img = img.unsqueeze(0).to(device)
imgGradient = imgGradient.unsqueeze(0).to(device)
net.eval()
with torch.no_grad():
    out = net(img, imgGradient)
predicted_class = out.argmax(dim=1).item()
print(test_dataset.tags[predicted_class])


# Step 6: Evaluate model accuracy

In [ ]:
def test(test_loader, net, criterion):
    losses = AverageMeter()
    top1   = AverageMeter()

    net.eval()

    for i, (img, imgGradient, target) in enumerate(test_loader, start=1):
        img = img.to(device)
        imgGradient = imgGradient.to(device)
        target = target.to(device)

        with torch.no_grad():
            out = net(img, imgGradient)
            loss = criterion(out, target)

        prec1 = accuracy(out, target) # prec1:list
        losses.update(loss.item(), img.size(0))
        top1.update(prec1.item(), img.size(0))

    f_l = [losses.avg, top1.avg]
    print('---------------------------------test classification result---------------------------------')
    print('Loss: {:.4f}, Prec@1: {:.2f}%'.format(*f_l))

    return top1.avg

test_top1 = test(test_loader, net, criterion)

# Step 7: T-SNE Visualization

##Use hooks in PyTorch to extract feature representations from the intermediate layers of the model for the test set "testloader", and visualize them using the T-SNE method. The specific requirements are as follows:
####Visualize the features before and after the dual-branch feature fusion. If there are multiple fusions, you may choose specific layers for visualization.
